In [6]:
!pip uninstall numpy -y
!pip install numpy --no-cache-dir

Found existing installation: numpy 2.4.2+computecanada
Uninstalling numpy-2.4.2+computecanada:
  Successfully uninstalled numpy-2.4.2+computecanada
Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic/numpy-2.4.2+computecanada-cp312-cp312-linux_x86_64.whl


In [7]:
import os
import json
import re
from collections import defaultdict
import numpy as np

BASE_DIR = "/home/piado/projects/aip-lindell/piado/vae/wandb_outputs/sweep"

STEPS = list(range(7000, 8001, 200))


def extract_config(config_path):
    with open(config_path, "r") as f:
        text = f.read()

    lora_rank_match = re.search(r"lora_rank\s*=\s*(\d+)", text)
    fusion_mode_match = re.search(r'fusion_mode\s*=\s*"([^"]+)"', text)

    lora_rank = int(lora_rank_match.group(1)) if lora_rank_match else None
    fusion_mode = fusion_mode_match.group(1) if fusion_mode_match else None

    return lora_rank, fusion_mode


def load_metrics(file_path):
    with open(file_path, "r") as f:
        data = json.load(f)

    columns = data["columns"]
    values = data["data"]

    idx = {col: i for i, col in enumerate(columns)}

    frames = defaultdict(lambda: {"psnr": [], "ssim": [], "mse": []})

    for row in values:
        frame = row[idx["frame"]]
        frames[frame]["psnr"].append(row[idx["psnr"]])
        frames[frame]["ssim"].append(row[idx["ssim"]])
        frames[frame]["mse"].append(row[idx["mse"]])

    return frames


results = []

for run in os.listdir(BASE_DIR):
    run_path = os.path.join(BASE_DIR, run)

    if not os.path.isdir(run_path):
        continue

    fixed_seq_dir = os.path.join(run_path, "media/table/fixed_seq")
    config_path = os.path.join(
        run_path,
        "outputs/cross_attention_rank64_viewwise_decoder/training_config_snapshot.py"
    )

    if not os.path.exists(fixed_seq_dir) or not os.path.exists(config_path):
        continue

    lora_rank, fusion_mode = extract_config(config_path)

    if lora_rank is None or fusion_mode is None:
        continue

    all_frames = defaultdict(lambda: {"psnr": [], "ssim": [], "mse": []})

    for step in STEPS:
        files = [
            f for f in os.listdir(fixed_seq_dir)
            if f.startswith(f"train_metrics_{step}_") and f.endswith(".json")
        ]

        for file in files:
            path = os.path.join(fixed_seq_dir, file)
            frame_data = load_metrics(path)

            for frame, metrics in frame_data.items():
                for k in metrics:
                    all_frames[frame][k].extend(metrics[k])

    # compute averages
    per_frame_avg = {}
    global_metrics = {"psnr": [], "ssim": [], "mse": []}

    for frame, metrics in all_frames.items():
        per_frame_avg[frame] = {
            k: np.mean(v) for k, v in metrics.items()
        }
        for k in metrics:
            global_metrics[k].extend(metrics[k])

    global_avg = {k: np.mean(v) for k, v in global_metrics.items()}

    results.append({
        "rank": lora_rank,
        "fusion_mode": fusion_mode,
        "per_frame": per_frame_avg,
        "global": global_avg
    })


# -------------------------
# Aggregation
# -------------------------

rank_scores = defaultdict(list)
fusion_scores = defaultdict(list)
rank_fusion_scores = defaultdict(lambda: defaultdict(list))

for r in results:
    rank = r["rank"]
    fusion = r["fusion_mode"]
    score = r["global"]["psnr"]  # use PSNR as main metric

    rank_scores[rank].append(score)
    fusion_scores[fusion].append(score)
    rank_fusion_scores[fusion][rank].append(score)


rank_avg = {k: np.mean(v) for k, v in rank_scores.items()}
fusion_avg = {k: np.mean(v) for k, v in fusion_scores.items()}
rank_fusion_avg = {
    f: {r: np.mean(v) for r, v in ranks.items()}
    for f, ranks in rank_fusion_scores.items()
}


# -------------------------
# Reporting
# -------------------------

best_rank = max(rank_avg, key=rank_avg.get)
best_fusion = max(fusion_avg, key=fusion_avg.get)

best_rank_per_fusion = {
    f: max(ranks, key=ranks.get)
    for f, ranks in rank_fusion_avg.items()
}

print("\n=== GLOBAL RESULTS ===")
print("Best Rank:", best_rank, rank_avg[best_rank])
print("Best Fusion Mode:", best_fusion, fusion_avg[best_fusion])

print("\n=== Rank per Fusion Mode ===")
for f, r in best_rank_per_fusion.items():
    print(f"{f}: best rank = {r}, score = {rank_fusion_avg[f][r]}")

ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python 3.12 from "/project/6101839/piado/.venv/bin/python"
  * The NumPy version is: "2.4.2"

and make sure that they are the versions you expect.

Please carefully study the information and documentation linked above.
This is unlikely to be a NumPy issue but will be caused by a bad install
or environment on your machine.

Original error was: libcpupower.so.0: cannot open shared object file: No such file or directory
